# Towards Generative Modeling of CO₂ Geological Storage with Latent Conditional Diffusion Models

|| **Author: Vittoria De Pellegrini** || **Year: 2025** ||
**Institution: King Abdullah University of Science and Technology (KAUST)** ||

This notebook provides a comprehensive guide for training a **Latent Conditional Diffusion Model** for Pressure BuildUp Dataset.

### ⚠️ Define Normalization Strategy

To normalize our dataset to the range $[-1, 1]$, we utilize **Local Normalization**. This approach involves normalizing each sample independently based on its own minimum and maximum values. We adopt local normalization because the range of values differs significantly between samples. The formula for local normalization is calculated as follows: 

$$x_{\text{norm}} = 2 \times \frac{x - \min(x)}{\max(x) - \min(x)} - 1\tag{1}$$

Where:
- $\min(x)$ and $\max(x)$ are the minimum and maximum values of the each sample $x$ in the dataset $X = \{x_1, x_2, \dots, x_N\}$, where $N$ is the number of samples.
- $x_{\text{norm}}$ denotes the normalized value, which will falls in the range $[-1, 1]$.

Note: 
- All 12 conditions are stored as 2D maps. If some conditions have fixed values, we do not apply any normalization. 

### 📚⬇ Import Libraries and Functions 

In [1]:
# Import libraries 
import warnings
warnings.filterwarnings('ignore')

import os
import json
import optuna 

import accelerate 
from accelerate import notebook_launcher

import torch
from torch import nn, optim
from torch.optim import Adam
from torchinfo import summary
from torch.utils.data import DataLoader, TensorDataset

import matplotlib.pyplot as plt

import sys
sys.path.append('/.../')

# Import functions 
from dmfco2 import(
    set_seed,   
    dnorm_dP,    
    Linear_Variance_Scheduler,
    Cosine_Variance_Scheduler,
    VQVAE,
    Conditional_Unet,
    trainer_diffusion,
    visualize_gaussian_forward_diffusion_dP_train,
    visualize_reverse_generation_dP_train
)

### ❗️ Identify Number of Devices (Multi-GPU Training with Accelerate)

In [2]:
# Print the number of GPUs available
num_devices = torch.cuda.device_count()
print("Number of devices: ", num_devices)

# Loop through all available devices and print their names
for i in range(num_devices):
    print(f"Device {i}: {torch.cuda.get_device_name(i)}")
    
# Print current device
print("Current device: ", torch.cuda.current_device())

### ❗️Set Seed for Reproducible Results

In [3]:
# Set seed
set_seed(seed=42)

### 📘 Define Inputs (Conditions) and Outputs (Targets) Variables for Pressure BuildUp Training Dataset

In [ ]:
# Define training dataset directory path
data_dir_dP_train = '/.../co2_dataset/training_dataset_pressure_buildup'

In [5]:
# Define inputs (conditions) and outputs (targets) variables
train_dP_inputs = torch.load(f'{data_dir_dP_train}/dP_train_inputs.pt')
train_dP_outputs = torch.load(f'{data_dir_dP_train}/dP_train_outputs.pt')

print(train_dP_inputs.shape) # Shape: [num_samples, height, width, time_steps, num_features]
print(train_dP_outputs.shape)

In [6]:
# Inputs -> Extract last time frame 30.0 y (index -1) and we consider all conditions which are 12 
train_dP_inputs  = train_dP_inputs[:, :, :, -1, :]  

# Outputs -> Extract last time frame 30.0 y
train_dP_outputs = train_dP_outputs[:, :, :, -1] 

print("New shape:", train_dP_inputs.shape)
print("New shape:", train_dP_outputs.shape)

In [7]:
# Add number of features (1) to outputs (targets)
train_dP_inputs = train_dP_inputs.permute(0, 3, 1, 2)  # Shape will be [4500, 12, 96, 200]
train_dP_outputs = train_dP_outputs[:, :, :].unsqueeze(1) # Shape will be [4500, 1, 96, 200]

# Print new shape
print("New shape:", train_dP_inputs.shape)   # Expected: [num_samples, num_features, height, width] -> All conditions
print("New shape:", train_dP_outputs.shape)  # Expected: [num_samples, num_features, height, width] -> Only Pressure Build-Up @ year 30.0

In [8]:
# Apply dnorm_dP function to ensure proper Pressure Build-Up range
train_dP_outputs=dnorm_dP(train_dP_outputs)

# Check Min and Max values 
min_train_dP = train_dP_outputs.min()
max_train_dP = train_dP_outputs.max()
print(min_train_dP) 
print(max_train_dP)

In [ ]:
# Some dP values are negative. Replace negative values with zeros
train_dP_outputs = torch.clamp(train_dP_outputs, min=0)

# Check Min and Max values 
min_train_dP = train_dP_outputs.min()
max_train_dP = train_dP_outputs.max()
print(min_train_dP) 
print(max_train_dP)

In [10]:
# Define file path to save the min-max values BEFORE local normalization for the conditions 
save_path = os.path.join(data_dir_dP_train, "dP_train_conditions_min_max_before_normalization.txt")

# Open the file for writing
with open(save_path, "w") as f:
    f.write("Condition Min-Max Values Per Sample\n")
    f.write("=" * 50 + "\n")

    num_samples = train_dP_inputs.shape[0]  # Number of samples
    num_conditions = min(train_dP_inputs.shape[-1], 12) # Number of conditions (12)

    for i in range(num_conditions):  
        f.write(f"\nCondition {i+1}:\n")  
        f.write("-" * 50 + "\n")

        # Extract condition `i`
        condition_i = train_dP_inputs[:, i, :, :]  # Shape: [4500, 96, 200]

        # Compute min and max values for each sample
        min_vals = condition_i.amin(dim=(1, 2))  # Min per sample
        max_vals = condition_i.amax(dim=(1, 2))  # Max per sample

        # Write per-sample min and max values
        for sample_idx in range(num_samples):
            min_val = min_vals[sample_idx].item()
            max_val = max_vals[sample_idx].item()

            # Write to the text file
            f.write(f"Sample {sample_idx+1}: Min: {min_val}, Max: {max_val}\n")

print(f"Condition min-max values saved in: {save_path}")

In [11]:
# Define file path to save the min-max values for porosity BEFORE local normalization
save_path = os.path.join(data_dir_dP_train, "dP_train_porosity_min_max_before_normalization.txt")

# Open the file for writing
with open(save_path, "w") as f:
    f.write("Porosity Min-Max Values Per Sample\n")
    f.write("=" * 50 + "\n")

    num_samples = train_dP_inputs.shape[0]  # Number of samples

    # Initialize lists to store min/max values for porosity (Condition Index 2)
    min_train_por = []
    max_train_por = []

    # Extract porosity condition (Index 2)
    porosity_condition = train_dP_inputs[:, 2, :, :]  # Shape: [4500, 96, 200]

    # Compute per-sample min and max values for porosity
    for i in range(num_samples):
        min_val = porosity_condition[i].amin().item()  # Min for sample i
        max_val = porosity_condition[i].amax().item()  # Max for sample i
        min_train_por.append(min_val)
        max_train_por.append(max_val)

        # Write only porosity min/max values to the file
        f.write(f"Sample {i+1}: Min Porosity = {min_val}, Max Porosity = {max_val}\n")

print(f"Porosity min-max values saved in: {save_path}")

In [12]:
# Define file path to save the min-max values AFTER local normalization
save_path = os.path.join(data_dir_dP_train, "dP_train_conditions_min_max_after_normalization.txt")

# Get shapes
num_samples = train_dP_inputs.shape[0]  # 4500
num_conditions = train_dP_inputs.shape[1]  # 12

# Local normalization: Normalize each sample's condition between -1 and 1
for i in range(num_conditions):
    condition_i = train_dP_inputs[:, i, :, :]  # Shape: [4500, 96, 200]

    # Compute per-sample min and max values
    min_vals = condition_i.amin(dim=(1, 2), keepdim=True)  # Shape: [4500, 1, 1]
    max_vals = condition_i.amax(dim=(1, 2), keepdim=True)  # Shape: [4500, 1, 1]

    # Compute denominator (range)
    diff = max_vals - min_vals

    # Fix: Handle constant conditions where min == max
    mask = diff == 0  # Identifies samples where max == min
    diff = diff.masked_fill(mask, 1)  # Prevent zero division

    # Normalize per sample
    normalized_condition = 2 * (condition_i - min_vals) / diff - 1

    # Keep original values when constant 
    normalized_condition.masked_scatter_(mask.expand_as(normalized_condition), condition_i)

    # Store back
    train_dP_inputs[:, i, :, :] = normalized_condition

# Save min-max values AFTER local normalization
with open(save_path, "w") as f:
    f.write("Condition Min-Max Values AFTER Local Normalization (Per Sample)\n")
    f.write("=" * 50 + "\n")

    for i in range(num_conditions):  
        f.write(f"\nCondition {i+1} (After Local Normalization):\n")  
        f.write("-" * 50 + "\n")

        condition_i = train_dP_inputs[:, i, :, :]
        min_vals = condition_i.amin(dim=(1, 2))
        max_vals = condition_i.amax(dim=(1, 2))

        for sample_idx in range(num_samples):
            min_val = min_vals[sample_idx].item()
            max_val = max_vals[sample_idx].item()
            f.write(f"Sample {sample_idx+1}: Min: {min_val}, Max: {max_val}\n")

print(f"Condition min-max values after local normalization saved in: {save_path}")

In [13]:
# Check min and max values after global normalization
# To understand if the variable is a 2D map or a scalar you need to check the min and max values
# If min and max values are constant then the variable is a scalar 
# If the min and max values are different then the variable is a 2D map 
print("\nChecking global min/max for each condition after normalization:")

for i in range(num_conditions):
    global_min = train_dP_inputs[:, i, :, :].amin().item()
    global_max = train_dP_inputs[:, i, :, :].amax().item()
    print(f"Condition {i+1}: Global Min = {global_min}, Global Max = {global_max}")

In [ ]:
# Check ranges for each sample individually (TARGET)
# Reduce over dimensions 1, 2, 3 to compute min and max for each sample
min_train_dP_individual = train_dP_outputs.amin(dim=(1, 2, 3))  # Min per sample
max_train_dP_individual = train_dP_outputs.amax(dim=(1, 2, 3))  # Max per sample

# Verify shapes
print(f"Shape of min_train_dP_individual: {min_train_dP_individual.shape}")  # [4500]
print(f"Shape of max_train_dP_individual: {max_train_dP_individual.shape}")  # [4500]

# Combine min and max into ranges
ranges = [(min_val.item(), max_val.item()) for min_val, max_val in zip(min_train_dP_individual, max_train_dP_individual)]

# Display all the ranges
for i, r in enumerate(ranges[:]): 
    print(f"Sample {i + 1} range: {r}")
    
# Check min and max values in train_dP_outputs before local normalization
print("Global Min for Pressure Build-Up before normalization:", train_dP_outputs.min().item(), "Global Max for Pressure Build-Up before normalization:", train_dP_outputs.max().item())

In [15]:
# Ensure logging directory exists
log_dir = "/.../co2_dataset/log_files/"
os.makedirs(log_dir, exist_ok=True)
dP_log_file = os.path.join(log_dir, "train_dP_min_max_before_normalization.txt")

# Save min and max values before normalization into a text file
with open(dP_log_file, "w") as log_file:
    log_file.write("Sample-wise Min-Max Values Before Normalization\n")
    log_file.write("=" * 80 + "\n")
    for i in range(len(min_train_dP_individual)):
        log_file.write(f"Sample {i+1}: Min Before={min_train_dP_individual[i].item():.4f}, Max Before={max_train_dP_individual[i].item():.4f}\n")

In [16]:
# Normalize outputs between -1 and 1 

# Initialize arrays to store min and max values for each sample
min_train_dP = []
max_train_dP = []

# Normalize outputs and store min/max values
for i in range(len(train_dP_outputs)):
    min_val = train_dP_outputs[i].min()
    max_val = train_dP_outputs[i].max()
    print(f"Sample {i + 1}:")
    print(f"Original Min: {min_val}, Original Max: {max_val}")
    min_train_dP.append(min_val)
    max_train_dP.append(max_val)

    # Local Normalization formula
    train_dP_outputs[i] = 2 * (train_dP_outputs[i] - min_val) / (max_val - min_val) - 1
    print(f"Normalized Output: {train_dP_outputs[i]}")

In [17]:
# Check ranges for each sample individually (TARGET)
# Reduce over dimensions 1, 2, 3 to compute min and max for each sample
min_train_dP_individual_after_norm = train_dP_outputs.amin(dim=(1, 2, 3))  # Min per sample
max_train_dP_individual_after_norm = train_dP_outputs.amax(dim=(1, 2, 3))  # Max per sample

# Verify shapes
print(f"Shape of min_train_dP_individual_after_norm: {min_train_dP_individual_after_norm.shape}")  # [4500]
print(f"Shape of max_train_dP_individual_after_norm: {max_train_dP_individual_after_norm.shape}")  # [4500]

# Combine min and max into ranges
ranges = [(min_val.item(), max_val.item()) for min_val, max_val in zip(min_train_dP_individual_after_norm, max_train_dP_individual_after_norm)]

# Display all the ranges
for i, r in enumerate(ranges[:]): 
    print(f"Sample {i + 1} range: {r}")
    
# Check min and max values in train_dP_outputs after local normalization
print("Global Min for Pressure Build-Up after normalization:", train_dP_outputs.min().item(), "Global Min for Pressure Build-Up after normalization:", train_dP_outputs.max().item())

In [18]:
# Ensure logging directory exists
log_dir = "/.../co2_dataset/log_files/"
os.makedirs(log_dir, exist_ok=True)
dP_log_file = os.path.join(log_dir, "train_dP_min_max_after_normalization.txt")

# Save min and max values before normalization into a text file
with open(dP_log_file, "w") as log_file:
    log_file.write("Sample-wise Min-Max Values After Normalization\n")
    log_file.write("=" * 80 + "\n")
    for i in range(len(min_train_dP_individual_after_norm)):
        log_file.write(f"Sample {i+1}: Min After={min_train_dP_individual_after_norm[i].item():.4f}, Max After={max_train_dP_individual_after_norm[i].item():.4f}\n")

In [ ]:
# Define variables for conditional diffusion model
train_dP_label = train_dP_inputs
train_dP_target = train_dP_outputs

### 📕 Define Inputs (Conditions) and Outputs (Targets) Variables for Pressure BuildUp Validation Dataset

In [ ]:
# Define validation dataset directory path
data_dir_dP_val = '/.../co2_dataset/validation_dataset_pressure_buildup'

In [21]:
# Define inputs (conditions) and outputs (targets) variables
val_dP_inputs = torch.load(f'{data_dir_dP_val}/dP_val_inputs.pt')
val_dP_outputs = torch.load(f'{data_dir_dP_val}/dP_val_outputs.pt')

print(val_dP_inputs.shape) # Shape: [num_samples, height, width, time_steps, num_features]
print(val_dP_outputs.shape)

In [ ]:
# Inputs -> Extract last time frame 30.0 y (index -1) and we consider all conditions which are 12 
val_dP_inputs  = val_dP_inputs[:, :, :, -1, :]  

# Outputs -> Extract last time frame 30.0 y
val_dP_outputs = val_dP_outputs[:, :, :, -1] 

print("New shape:", val_dP_inputs.shape)
print("New shape:", val_dP_outputs.shape)

In [23]:
# Add number of features (1) to outputs (targets)
val_dP_inputs = val_dP_inputs.permute(0, 3, 1, 2)  # Shape will be [500, 12, 96, 200]
val_dP_outputs = val_dP_outputs[:, :, :].unsqueeze(1) # Shape will be [500, 1, 96, 200]

# Print new shape
print("New shape:", val_dP_inputs.shape)   # Expected: [num_samples, num_features, height, width] -> All conditions
print("New shape:", val_dP_outputs.shape)  # Expected: [num_samples, num_features, height, width] -> Only Pressure Build-Up @ year 30.0

In [ ]:
# Apply dnorm_dP function to ensure proper Pressure Build-Up range
val_dP_outputs=dnorm_dP(val_dP_outputs)

# Check Min and Max values 
min_val_dP = val_dP_outputs.min()
max_val_dP = val_dP_outputs.max()
print(min_val_dP) 
print(max_val_dP)

In [25]:
# Some dP values are negative. Replace negative values with zeros
val_dP_outputs = torch.clamp(val_dP_outputs, min=0)

# Check Min and Max values 
min_val_dP = val_dP_outputs.min()
max_val_dP = val_dP_outputs.max()
print(min_val_dP) 
print(max_val_dP)

In [26]:
# Define file path to save the min-max values BEFORE local normalization for the conditions 
save_path = os.path.join(data_dir_dP_val, "dP_val_conditions_min_max_before_normalization.txt")

# Open the file for writing
with open(save_path, "w") as f:
    f.write("Condition Min-Max Values Per Sample\n")
    f.write("=" * 50 + "\n")

    num_samples = val_dP_inputs.shape[0]  # Number of samples
    num_conditions = min(val_dP_inputs.shape[-1], 12) # Number of conditions (12)

    for i in range(num_conditions):  
        f.write(f"\nCondition {i+1}:\n")  
        f.write("-" * 50 + "\n")

        # Extract condition `i`
        condition_i = val_dP_inputs[:, i, :, :]  # Shape: [500, 96, 200]

        # Compute min and max values for each sample
        min_vals = condition_i.amin(dim=(1, 2))  # Min per sample
        max_vals = condition_i.amax(dim=(1, 2))  # Max per sample

        # Write per-sample min and max values
        for sample_idx in range(num_samples):
            min_val = min_vals[sample_idx].item()
            max_val = max_vals[sample_idx].item()

            # Write to the text file
            f.write(f"Sample {sample_idx+1}: Min: {min_val}, Max: {max_val}\n")

print(f"Condition min-max values saved in: {save_path}")

In [27]:
# Define file path to save the min-max values for porosity BEFORE local normalization
save_path = os.path.join(data_dir_dP_val, "dP_val_porosity_min_max_before_normalization.txt")

# Open the file for writing
with open(save_path, "w") as f:
    f.write("Porosity Min-Max Values Per Sample\n")
    f.write("=" * 50 + "\n")

    num_samples = val_dP_inputs.shape[0]  # Number of samples

    # Initialize lists to store min/max values for porosity (Condition Index 2)
    min_val_por = []
    max_val_por = []

    # Extract porosity condition (Index 2)
    porosity_condition = val_dP_inputs[:, 2, :, :]  # Shape: [4500, 96, 200]

    # Compute per-sample min and max values for porosity
    for i in range(num_samples):
        min_val = porosity_condition[i].amin().item()  # Min for sample i
        max_val = porosity_condition[i].amax().item()  # Max for sample i
        min_val_por.append(min_val)
        max_val_por.append(max_val)

        # Write only porosity min/max values to the file
        f.write(f"Sample {i+1}: Min Porosity = {min_val}, Max Porosity = {max_val}\n")

print(f"Porosity min-max values saved in: {save_path}")

In [28]:
# Define file path to save the min-max values AFTER local normalization
save_path = os.path.join(data_dir_dP_val, "dP_val_conditions_min_max_after_normalization.txt")

# Get shapes
num_samples = val_dP_inputs.shape[0]  # 4500
num_conditions = val_dP_inputs.shape[1]  # 12

# Local normalization: Normalize each sample's condition between -1 and 1
for i in range(num_conditions):
    condition_i = val_dP_inputs[:, i, :, :]  # Shape: [4500, 96, 200]

    # Compute per-sample min and max values
    min_vals = condition_i.amin(dim=(1, 2), keepdim=True)  # Shape: [4500, 1, 1]
    max_vals = condition_i.amax(dim=(1, 2), keepdim=True)  # Shape: [4500, 1, 1]

    # Compute denominator (range)
    diff = max_vals - min_vals

    # Fix: Handle constant conditions where min == max
    mask = diff == 0  # Identifies samples where max == min
    diff = diff.masked_fill(mask, 1)  # Prevent zero division

    # Normalize per sample
    normalized_condition = 2 * (condition_i - min_vals) / diff - 1

    # Keep original values when constant 
    normalized_condition.masked_scatter_(mask.expand_as(normalized_condition), condition_i)

    # Store back
    val_dP_inputs[:, i, :, :] = normalized_condition

# Save min-max values AFTER local normalization
with open(save_path, "w") as f:
    f.write("Condition Min-Max Values AFTER Local Normalization (Per Sample)\n")
    f.write("=" * 50 + "\n")

    for i in range(num_conditions):  
        f.write(f"\nCondition {i+1} (After Local Normalization):\n")  
        f.write("-" * 50 + "\n")

        condition_i = val_dP_inputs[:, i, :, :]
        min_vals = condition_i.amin(dim=(1, 2))
        max_vals = condition_i.amax(dim=(1, 2))

        for sample_idx in range(num_samples):
            min_val = min_vals[sample_idx].item()
            max_val = max_vals[sample_idx].item()
            f.write(f"Sample {sample_idx+1}: Min: {min_val}, Max: {max_val}\n")

print(f"Condition min-max values after local normalization saved in: {save_path}")

In [29]:
# Check min and max values after global normalization
# To understand if the variable is a 2D map or a scalar you need to check the min and max values
# If min and max values are constant then the variable is a scalar 
# If the min and max values are different then the variable is a 2D map 
print("\nChecking global min/max for each condition after normalization:")

for i in range(num_conditions):
    global_min = val_dP_inputs[:, i, :, :].amin().item()
    global_max = val_dP_inputs[:, i, :, :].amax().item()
    print(f"Condition {i+1}: Global Min = {global_min}, Global Max = {global_max}")

In [30]:
# Check ranges for each sample individually (TARGET)
# Reduce over dimensions 1, 2, 3 to compute min and max for each sample
min_val_dP_individual = val_dP_outputs.amin(dim=(1, 2, 3))  # Min per sample
max_val_dP_individual = val_dP_outputs.amax(dim=(1, 2, 3))  # Max per sample

# Verify shapes
print(f"Shape of min_val_dP_individual: {min_val_dP_individual.shape}")  # [500]
print(f"Shape of max_val_dP_individual: {max_val_dP_individual.shape}")  # [500]

# Combine min and max into ranges
ranges = [(min_val.item(), max_val.item()) for min_val, max_val in zip(min_val_dP_individual, max_val_dP_individual)]

# Display all the ranges
for i, r in enumerate(ranges[:]): 
    print(f"Sample {i + 1} range: {r}")
    
# Check min and max values in val_dP_outputs before local normalization
print("Global Min for Pressure Build-Up before normalization:", val_dP_outputs.min().item(), "Global Max for Pressure Build-Up before normalization:", val_dP_outputs.max().item())

In [ ]:
# Ensure logging directory exists
log_dir = "/.../co2_dataset/log_files/"
os.makedirs(log_dir, exist_ok=True)
dP_log_file = os.path.join(log_dir, "val_dP_min_max_before_normalization.txt")

# Save min and max values before normalization into a text file
with open(dP_log_file, "w") as log_file:
    log_file.write("Sample-wise Min-Max Values Before Normalization\n")
    log_file.write("=" * 80 + "\n")
    for i in range(len(min_val_dP_individual)):
        log_file.write(f"Sample {i+1}: Min Before={min_val_dP_individual[i].item():.4f}, Max Before={max_val_dP_individual[i].item():.4f}\n")

In [32]:
# Normalize outputs between -1 and 1 

# Initialize arrays to store min and max values for each sample
min_val_dP = []
max_val_dP = []

# Normalize outputs and store min/max values
for i in range(len(val_dP_outputs)):
    min_val = val_dP_outputs[i].min()
    max_val = val_dP_outputs[i].max()
    print(f"Sample {i + 1}:")
    print(f"Original Min: {min_val}, Original Max: {max_val}")
    min_val_dP.append(min_val)
    max_val_dP.append(max_val)

    # Local Normalization formula
    val_dP_outputs[i] = 2 * (val_dP_outputs[i] - min_val) / (max_val - min_val) - 1
    print(f"Normalized Output: {val_dP_outputs[i]}")

In [33]:
# Check ranges for each sample individually (TARGET)
# Reduce over dimensions 1, 2, 3 to compute min and max for each sample
min_val_dP_individual_after_norm = val_dP_outputs.amin(dim=(1, 2, 3))  # Min per sample
max_val_dP_individual_after_norm = val_dP_outputs.amax(dim=(1, 2, 3))  # Max per sample

# Verify shapes
print(f"Shape of min_val_dP_individual_after_norm: {min_val_dP_individual_after_norm.shape}")  # [500]
print(f"Shape of max_val_dP_individual_after_norm: {max_val_dP_individual_after_norm.shape}")  # [500]

# Combine min and max into ranges
ranges = [(min_val.item(), max_val.item()) for min_val, max_val in zip(min_val_dP_individual_after_norm, max_val_dP_individual_after_norm)]

# Display all the ranges
for i, r in enumerate(ranges[:]): 
    print(f"Sample {i + 1} range: {r}")
    
# Check min and max values in val_dP_outputs after local normalization
print("Global Min for Pressure Build-Up after normalization:", val_dP_outputs.min().item(), "Global Min for Pressure Build-Up after normalization:", val_dP_outputs.max().item())

In [ ]:
# Ensure logging directory exists
log_dir = "/.../co2_dataset/log_files/"
os.makedirs(log_dir, exist_ok=True)
dP_log_file = os.path.join(log_dir, "val_dP_min_max_after_normalization.txt")

# Save min and max values before normalization into a text file
with open(dP_log_file, "w") as log_file:
    log_file.write("Sample-wise Min-Max Values After Normalization\n")
    log_file.write("=" * 80 + "\n")
    for i in range(len(min_val_dP_individual_after_norm)):
        log_file.write(f"Sample {i+1}: Min After={min_val_dP_individual_after_norm[i].item():.4f}, Max After={max_val_dP_individual_after_norm[i].item():.4f}\n")

In [ ]:
# Define variables for conditional diffusion model
val_dP_label = val_dP_inputs
val_dP_target = val_dP_outputs

### 📙 Define Inputs (Conditions) and Outputs (Targets) Variables for Pressure BuildUp Test Dataset

In [ ]:
# Define validation dataset directory path
data_dir_dP_test = '/.../co2_dataset/test_dataset_pressure_buildup'

In [37]:
# Define inputs (conditions) and outputs (targets) variables
test_dP_inputs = torch.load(f'{data_dir_dP_test}/dP_test_inputs.pt')
test_dP_outputs = torch.load(f'{data_dir_dP_test}/dP_test_outputs.pt')

print(test_dP_inputs.shape) # Shape: [num_samples, height, width, time_steps, num_features]
print(test_dP_outputs.shape)

In [ ]:
# Inputs -> Extract last time frame 30.0 y (index -1) and we consider all conditions which are 12 
test_dP_inputs  = test_dP_inputs[:, :, :, -1, :]  

# Outputs -> Extract last time frame 30.0 y
test_dP_outputs = test_dP_outputs[:, :, :, -1] 

print("New shape:", test_dP_inputs.shape)
print("New shape:", test_dP_outputs.shape)

In [39]:
# Add number of features (1) to inputs (conditions) and outputs (targets)
test_dP_inputs = test_dP_inputs.permute(0, 3, 1, 2) # Shape will be [500, 12, 96, 200]
test_dP_outputs = test_dP_outputs[:, :, :].unsqueeze(1) # Shape will be [500, 1, 96, 200]

# Print new shape
print("New shape:", test_dP_inputs.shape)   # Expected: [num_samples, num_features, height, width] -> All conditions
print("New shape:", test_dP_outputs.shape)  # Expected: [num_samples, num_features, height, width] -> Only Pressure Build-Up @ year 30.0

In [40]:
## FOR THE TEST DATASET WE DO NOT APPLY THE dnorm_dP FUNCTION

# Apply dnorm_dP function to ensure proper Pressure Build-Up range
#test_dP_outputs=dnorm_dP(test_dP_outputs)

# Check Min and Max values 
#min_test_dP = test_dP_outputs.min()
#max_test_dP = test_dP_outputs.max()
#print(min_test_dP) 
#print(max_test_dP)

In [41]:
# Some dP values are negative. Replace negative values with zeros
test_dP_outputs = torch.clamp(test_dP_outputs, min=0)

# Check Min and Max values 
min_test_dP = test_dP_outputs.min()
max_test_dP = test_dP_outputs.max()
print(min_test_dP) 
print(max_test_dP)

In [42]:
# Define file path to save the min-max values BEFORE local normalization for the conditions 
save_path = os.path.join(data_dir_dP_test, "dP_test_conditions_min_max_before_normalization.txt")

# Open the file for writing
with open(save_path, "w") as f:
    f.write("Condition Min-Max Values Per Sample\n")
    f.write("=" * 50 + "\n")

    num_samples = test_dP_inputs.shape[0]  # Number of samples
    num_conditions = min(test_dP_inputs.shape[-1], 12) # Number of conditions (12)

    for i in range(num_conditions):  
        f.write(f"\nCondition {i+1}:\n")  
        f.write("-" * 50 + "\n")

        # Extract condition `i`
        condition_i = test_dP_inputs[:, i, :, :]  # Shape: [500, 96, 200]

        # Compute min and max values for each sample
        min_vals = condition_i.amin(dim=(1, 2))  # Min per sample
        max_vals = condition_i.amax(dim=(1, 2))  # Max per sample

        # Write per-sample min and max values
        for sample_idx in range(num_samples):
            min_val = min_vals[sample_idx].item()
            max_val = max_vals[sample_idx].item()

            # Write to the text file
            f.write(f"Sample {sample_idx+1}: Min: {min_val}, Max: {max_val}\n")

print(f"Condition min-max values saved in: {save_path}")

In [43]:
# Define file path to save the min-max values for porosity BEFORE local normalization
save_path = os.path.join(data_dir_dP_test, "dP_test_porosity_min_max_before_normalization.txt")

# Open the file for writing
with open(save_path, "w") as f:
    f.write("Porosity Min-Max Values Per Sample\n")
    f.write("=" * 50 + "\n")

    num_samples = test_dP_inputs.shape[0]  # Number of samples

    # Initialize lists to store min/max values for porosity (Condition Index 2)
    min_test_por = []
    max_test_por = []

    # Extract porosity condition (Index 2)
    porosity_condition = test_dP_inputs[:, 2, :, :]  # Shape: [4500, 96, 200]

    # Compute per-sample min and max values for porosity
    for i in range(num_samples):
        min_val = porosity_condition[i].amin().item()  # Min for sample i
        max_val = porosity_condition[i].amax().item()  # Max for sample i
        min_test_por.append(min_val)
        max_test_por.append(max_val)

        # Write only porosity min/max values to the file
        f.write(f"Sample {i+1}: Min Porosity = {min_val}, Max Porosity = {max_val}\n")

print(f"Porosity min-max values saved in: {save_path}")

In [44]:
# Define file path to save the min-max values AFTER local normalization
save_path = os.path.join(data_dir_dP_test, "dP_test_conditions_min_max_after_normalization.txt")

# Get shapes
num_samples = test_dP_inputs.shape[0]  # 4500
num_conditions = test_dP_inputs.shape[1]  # 12

# Local normalization: Normalize each sample's condition between -1 and 1
for i in range(num_conditions):
    condition_i = test_dP_inputs[:, i, :, :]  # Shape: [4500, 96, 200]

    # Compute per-sample min and max values
    min_vals = condition_i.amin(dim=(1, 2), keepdim=True)  # Shape: [4500, 1, 1]
    max_vals = condition_i.amax(dim=(1, 2), keepdim=True)  # Shape: [4500, 1, 1]

    # Compute denominator (range)
    diff = max_vals - min_vals

    # Fix: Handle constant conditions where min == max
    mask = diff == 0  # Identifies samples where max == min
    diff = diff.masked_fill(mask, 1)  # Prevent zero division

    # Normalize per sample
    normalized_condition = 2 * (condition_i - min_vals) / diff - 1

    # Keep original values when constant 
    normalized_condition.masked_scatter_(mask.expand_as(normalized_condition), condition_i)

    # Store back
    test_dP_inputs[:, i, :, :] = normalized_condition

# Save min-max values AFTER local normalization
with open(save_path, "w") as f:
    f.write("Condition Min-Max Values AFTER Local Normalization (Per Sample)\n")
    f.write("=" * 50 + "\n")

    for i in range(num_conditions):  
        f.write(f"\nCondition {i+1} (After Local Normalization):\n")  
        f.write("-" * 50 + "\n")

        condition_i = test_dP_inputs[:, i, :, :]
        min_vals = condition_i.amin(dim=(1, 2))
        max_vals = condition_i.amax(dim=(1, 2))

        for sample_idx in range(num_samples):
            min_val = min_vals[sample_idx].item()
            max_val = max_vals[sample_idx].item()
            f.write(f"Sample {sample_idx+1}: Min: {min_val}, Max: {max_val}\n")

print(f"Condition min-max values after local normalization saved in: {save_path}")

In [45]:
# Check min and max values after global normalization
# To understand if the variable is a 2D map or a scalar you need to check the min and max values
# If min and max values are constant then the variable is a scalar 
# If the min and max values are different then the variable is a 2D map 
print("\nChecking global min/max for each condition after normalization:")

for i in range(num_conditions):
    global_min = test_dP_inputs[:, i, :, :].amin().item()
    global_max = test_dP_inputs[:, i, :, :].amax().item()
    print(f"Condition {i+1}: Global Min = {global_min}, Global Max = {global_max}")

In [46]:
# Check ranges for each sample individually (TARGET)
# Reduce over dimensions 1, 2, 3 to compute min and max for each sample
min_test_dP_individual = test_dP_outputs.amin(dim=(1, 2, 3))  # Min per sample
max_test_dP_individual = test_dP_outputs.amax(dim=(1, 2, 3))  # Max per sample

# Verify shapes
print(f"Shape of min_test_dP_individual: {min_test_dP_individual.shape}")  # [500]
print(f"Shape of max_test_dP_individual: {max_test_dP_individual.shape}")  # [500]

# Combine min and max into ranges
ranges = [(min_val.item(), max_val.item()) for min_val, max_val in zip(min_test_dP_individual, max_test_dP_individual)]

# Display all the ranges
for i, r in enumerate(ranges[:]): 
    print(f"Sample {i + 1} range: {r}")
    
# Check min and max values in test_dP_outputs before local normalization
print("Global Min for Pressure Build-Up before normalization:", test_dP_outputs.min().item(), "Global Max for Pressure Build-Up before normalization:", test_dP_outputs.max().item())

In [47]:
# Ensure logging directory exists
log_dir = "/.../co2_dataset/log_files/"
os.makedirs(log_dir, exist_ok=True)
dP_log_file = os.path.join(log_dir, "test_dP_min_max_before_normalization.txt")

# Save min and max values before normalization into a text file
with open(dP_log_file, "w") as log_file:
    log_file.write("Sample-wise Min-Max Values Before Normalization\n")
    log_file.write("=" * 80 + "\n")
    for i in range(len(min_test_dP_individual)):
        log_file.write(f"Sample {i+1}: Min Before={min_test_dP_individual[i].item():.4f}, Max Before={max_test_dP_individual[i].item():.4f}\n")

In [48]:
# Normalize outputs between -1 and 1 

# Initialize arrays to store min and max values for each sample
min_test_dP = []
max_test_dP = []

# Normalize outputs and store min/max values
for i in range(len(test_dP_outputs)):
    min_val = test_dP_outputs[i].min()
    max_val = test_dP_outputs[i].max()
    print(f"Sample {i + 1}:")
    print(f"Original Min: {min_val}, Original Max: {max_val}")
    min_test_dP.append(min_val)
    max_test_dP.append(max_val)

    # Local Normalization formula
    test_dP_outputs[i] = 2 * (test_dP_outputs[i] - min_val) / (max_val - min_val) - 1
    print(f"Normalized Output: {test_dP_outputs[i]}")

In [49]:
# Check ranges for each sample individually (TARGET)
# Reduce over dimensions 1, 2, 3 to compute min and max for each sample
min_test_dP_individual_after_norm = test_dP_outputs.amin(dim=(1, 2, 3))  # Min per sample
max_test_dP_individual_after_norm = test_dP_outputs.amax(dim=(1, 2, 3))  # Max per sample

# Verify shapes
print(f"Shape of min_test_dP_individual_after_norm: {min_test_dP_individual_after_norm.shape}")  # [500]
print(f"Shape of max_test_dP_individual_after_norm: {max_test_dP_individual_after_norm.shape}")  # [500]

# Combine min and max into ranges
ranges = [(min_val.item(), max_val.item()) for min_val, max_val in zip(min_test_dP_individual_after_norm, max_test_dP_individual_after_norm)]

# Display all the ranges
for i, r in enumerate(ranges[:]): 
    print(f"Sample {i + 1} range: {r}")
    
# Check min and max values in test_dP_outputs after local normalization
print("Global Min for Pressure Build-Up after normalization:", test_dP_outputs.min().item(), "Global Min for Pressure Build-Up after normalization:", test_dP_outputs.max().item())

In [50]:
# Ensure logging directory exists
log_dir = "/.../co2_dataset/log_files/"
os.makedirs(log_dir, exist_ok=True)
dP_log_file = os.path.join(log_dir, "test_dP_min_max_after_normalization.txt")

# Save min and max values before normalization into a text file
with open(dP_log_file, "w") as log_file:
    log_file.write("Sample-wise Min-Max Values After Normalization\n")
    log_file.write("=" * 80 + "\n")
    for i in range(len(min_test_dP_individual_after_norm)):
        log_file.write(f"Sample {i+1}: Min After={min_test_dP_individual_after_norm[i].item():.4f}, Max After={max_test_dP_individual_after_norm[i].item():.4f}\n")

In [ ]:
# Define variables for conditional diffusion model
test_dP_label = test_dP_inputs
test_dP_target = test_dP_outputs

### 🗂️ Define Training, Validation, Test Datasets and Dataloaders

In [52]:
# Create dataloaders with augmentation 
training_dataset = TensorDataset(train_dP_target, train_dP_label)
validation_dataset = TensorDataset(val_dP_target, val_dP_label)
testing_dataset  = TensorDataset(test_dP_target, test_dP_label)

batch_size = 32

training_dataloader = DataLoader(training_dataset, 
                                 batch_size=batch_size, 
                                 shuffle=True,
                                 drop_last=True, # To remove the batch with less samples
                                 pin_memory=True, # Optimizes data transfer to GPU
                                 #num_workers=4 # Parallel data loading
                                 ) 

validation_dataloader = DataLoader(validation_dataset, 
                                   batch_size=batch_size, 
                                   shuffle=False,
                                   drop_last=True, # To remove the batch with less samples
                                   pin_memory=True, # Optimizes data transfer to GPU
                                   #num_workers=4 # Parallel data loading
                                   ) 

testing_dataloader = DataLoader(testing_dataset, 
                                batch_size=batch_size, 
                                shuffle=False,
                                drop_last=True, # To remove the batch with less samples
                                pin_memory=True, # Optimizes data transfer to GPU
                                #num_workers=4 # Parallel data loading
                                )  

# Check the data sizes
for feat, lab in training_dataloader:
    print(f'Features shape: {feat.shape} \t Labels shape: {lab.shape}')
    break

# Check the data sizes
for feat, lab in validation_dataloader:
    print(f'Features shape: {feat.shape} \t Labels shape: {lab.shape}')
    break

# Check the data sizes
for feat, lab in testing_dataloader:
    print(f'Features shape: {feat.shape} \t Labels shape: {lab.shape}')
    break

### 🚀 Preparing for Diffusion Model Training with Optuna and Accelerator: Pressure Build-Up

#### 01. Linear and Cosine Variance Scheduler Plots

In [53]:
# Define parameters 
T = 1000 # Time steps
beta_start=0.0001
beta_end=0.02

# Save directory
save_directory = '/.../trained_models/pressure_buildup_models'

# Instantiate the scheduler
Linear_Scheduler = Linear_Variance_Scheduler(time_steps=T, beta_start=beta_start, beta_end=beta_end)
Cosine_Scheduler = Cosine_Variance_Scheduler(time_steps=T, s=0.008)

# Plot the scheduler
Linear_Scheduler.plot_linear_variance_schedule(time_steps=T, beta_start=beta_start, beta_end=beta_end, save_dir=save_directory)
Cosine_Scheduler.plot_cosine_variance_schedule(time_steps=T, s=0.008,  save_dir=save_directory)

#### 02. Preliminary Training with Optuna and Accelerator
The goal is to identify the optimal combination of hyperparameters using Optuna for efficient hyperparameter tuning, and Accelerator to speed up training time. For quick evaluation we use few epochs.

In [ ]:
# Define the objective function for Optuna hyperparameter optimization
def objective_diffusion(trial):
    
    # Define hyperparameter search space
    ddpm = trial.suggest_categorical("ddpm", ["linear", "cosine"])
    n_epochs = trial.suggest_categorical("n_epochs", [10])  # Fewer epochs for quick evaluation
    learning_rate = trial.suggest_float("learning_rate", 1e-6, 2e-4, log=True) # search on a logarithmic scale

    # Define DDPM scheduler
    if ddpm == "linear":
        T = 1000
        beta_start = 0.0001
        beta_end = 0.02
        ddpm = Linear_Variance_Scheduler(time_steps=T, beta_start=beta_start, beta_end=beta_end)
    elif ddpm == "cosine":
        T = 1000
        ddpm = Cosine_Variance_Scheduler(time_steps=T, s=0.008)

    # Print trial details 
    print(f"Trial {trial.number}: Parameters - {trial.params}")


    # Load pre-trained VQVAE checkpoint
    vqvae_checkpoint = '/.../trained_models/pressure_buildup_models/vqvae/best_model/vqvae_model_partial_training.pth'
    vqvae_checkpoint = torch.load(vqvae_checkpoint)
    # Extract parameters and weights
    z_channels = vqvae_checkpoint["z_channels"]
    codebook_size = vqvae_checkpoint["codebook_size"]
    vqvae_state_dict = vqvae_checkpoint["state_dict"]
    # Initialize the model with the saved parameters
    vqvae_model = VQVAE(z_channels=z_channels, codebook_size=codebook_size)
    # Load the model weights
    vqvae_model.load_state_dict(vqvae_state_dict)
    # Print successful loading
    print("Model loaded successfully with z_channels={} and codebook_size={}".format(z_channels, codebook_size))
    
    # Initialize diffusion model
    model_unet = Conditional_Unet(t_emb_dim=256, z_channels=z_channels, image_cond=True, scalar_cond=False)
    
    # Visualize the Unet architecture
    summary(Conditional_Unet(t_emb_dim=256, z_channels=z_channels, image_cond=True, scalar_cond=False).to('cpu'), input_size=[(batch_size, z_channels, 3, 6), (batch_size,), (batch_size, 12, 96, 200)], dtypes=[torch.float, torch.float, torch.float])
    
    # Define optimizer
    optimizer = torch.optim.Adam(model_unet.parameters(), lr=learning_rate)

    # Define criterion
    criterion = nn.MSELoss()

    # Directory for trial results
    trial_save_dir = f"/.../trained_models/pressure_buildup_models/diffusion_model/trial_model_{trial.number}"
    os.makedirs(trial_save_dir, exist_ok=True)
    summary_file_path = os.path.join(trial_save_dir, "diffusion_training_summary.txt")

    # Conduct partial training for the trial
    training_loss, validation_loss, test_loss = trainer_diffusion(
        model_unet=model_unet,
        vqvae_model=vqvae_model,
        ddpm=ddpm,
        criterion=criterion,
        optimizer=optimizer,
        training_dataloader=training_dataloader,
        validation_dataloader=validation_dataloader,
        testing_dataloader=testing_dataloader,
        n_epochs=n_epochs,
        save_dir=trial_save_dir,
        save_dir_results=None,
        summary_file_path=summary_file_path,
        trial_params=trial.params,
        save_loss_curves=False,
        lambda_pixel=1.0,
        early_stopping_patience=10   
    )

    # Return validation loss at the end of training as the optimization metric
    return validation_loss[-1]

In [ ]:
# Save Best Hyperparameters into JSON file 

# Define the storage for Optuna to enable dashboard tracking
storage_url = "sqlite:////.../trained_models/pressure_buildup_models/diffusion_model/optuna_diffusion_model.db"

# Define file paths
best_trial_dir = "/.../trained_models/pressure_buildup_models/diffusion_model/best_model"
best_trial_file = os.path.join(best_trial_dir, "best_trial_params_partial_training.json")

# Ensure the directory exists
os.makedirs(best_trial_dir, exist_ok=True)

# Define total trials (make sure this is your actual target)
total_trials = 100 

# Define the optimization process
def optimize_diffusion():
    global study_diffusion
    study_diffusion = optuna.create_study(direction="minimize", study_name="diffusion_model_hyperparam_tuning", storage=storage_url, load_if_exists=True, sampler=optuna.samplers.TPESampler()) # Minimize Validation Loss

    # Check completed trials
    completed_trials = len(study_diffusion.trials)
    remaining_trials = max(0, total_trials - completed_trials)  # Avoid negative values

    print(f"Total Trials Target: {total_trials}")
    print(f"Completed Trials: {completed_trials}")
    print(f"Remaining Trials: {remaining_trials}")

    if remaining_trials > 0:
        # Define callback to save the best trial parameters after each trial
        def save_best_trial(study_diffusion, trial):
            best_trial = study_diffusion.best_trial
            print(f"Best trial number: {best_trial.number}")
            print(f"Best trial hyperparameters: {best_trial.params}")
            
            # Save the best trial parameters to a file
            with open(best_trial_file, "w") as f:
                json.dump(best_trial.params, f)
            
            print(f"Best trial parameters (Trial {best_trial.number}) saved to {best_trial_file}")

        study_diffusion.optimize(objective_diffusion, n_trials=remaining_trials, callbacks=[save_best_trial])
    else:
        print("All trials are already completed. No additional trials will be run.")

# Launch optimization process (for multi-GPU setups, adjust num_processes)
notebook_launcher(optimize_diffusion, args=(), num_processes=1)

### How to Activate Optuna Dashboard ###
#Copy paste this line into your terminal: optuna-dashboard sqlite:///optuna_diffusion_model.db
#Directly copy paste into your terminal: optuna-dashboard sqlite:////.../trained_models/pressure_buildup_models/diffusion_model/optuna_diffusion_model.db
#Go to VSC Command Palette and then “Simple Browser: Show” and copy paste the URL link: http://127.0.0.1:8080

#### 03. Full Training with Optuna and Accelerator

After identifying the best hyperparameters, we train the diffusion model on an increased number of epochs. 

##### 03a. Set-Up Full Training

In [ ]:
# Load best trial parameters
# File path to save the best trial parameters
best_trial_file = "/.../trained_models/pressure_buildup_models/diffusion_model/best_model/best_trial_params_partial_training.json"
with open(best_trial_file, "r") as f:
    best_trial_params = json.load(f)
      
# Define hyperparameter search space and learning rate
best_ddpm = best_trial_params["ddpm"]
best_learning_rate = best_trial_params["learning_rate"]

print("Best trial parameters loaded successfully.")

# Print best hyperparameters
print(f"Best ddpm: {best_ddpm}")
print(f"Best learning_rate: {best_learning_rate}")

# Define number of epochs
n_epochs=300

# Load pre-trained VQVAE checkpoint
vqvae_checkpoint = '/.../trained_models/pressure_buildup_models/vqvae/best_model/vqvae_model.pth'
vqvae_checkpoint = torch.load(vqvae_checkpoint)
# Extract parameters and weights
z_channels = vqvae_checkpoint["z_channels"]
codebook_size = vqvae_checkpoint["codebook_size"]
vqvae_state_dict = vqvae_checkpoint["state_dict"]
# Initialize the model with the saved parameters
vqvae_model = VQVAE(z_channels=z_channels, codebook_size=codebook_size)
# Load the model weights
vqvae_model.load_state_dict(vqvae_state_dict)
# Print successful loading
print("Model loaded successfully with z_channels={} and codebook_size={}".format(z_channels, codebook_size))
    
# Initialize diffusion model
model_unet = Conditional_Unet(t_emb_dim=256, z_channels=z_channels, image_cond=True, scalar_cond=False)
    
# Visualize the Unet architecture
summary(Conditional_Unet(t_emb_dim=256, z_channels=z_channels, image_cond=True, scalar_cond=False).to('cpu'), 
        input_size=[(batch_size, z_channels, 3, 6), # pressure buildup shape in the latent space 
                    (batch_size,), # Time embedding shape
                    (batch_size, 12, 96, 200)], # Condition/label shape
        dtypes=[torch.float, torch.float, torch.float])
    
# Define optimizer
optimizer = torch.optim.Adam(model_unet.parameters(), lr=best_learning_rate)

# Define criterion
criterion = nn.MSELoss()

# Define parameters and scheduler 
T = 1000 # Time steps
beta_start=0.0001
beta_end=0.02

if best_ddpm == "linear":
    ddpm = Linear_Variance_Scheduler(time_steps=T, beta_start=beta_start, beta_end=beta_end)
elif best_ddpm == "cosine":
    ddpm = Cosine_Variance_Scheduler(time_steps=T, s=0.008)
else:
    raise ValueError(f"Unknown ddpm type: {best_ddpm}")

# Directory for final model and results
final_save_dir = "/.../trained_models/pressure_buildup_models/diffusion_model/best_model"
os.makedirs(final_save_dir, exist_ok=True)
final_summary_path = "/.../trained_models/pressure_buildup_models/diffusion_model/best_model/diffusion_training_summary_full.txt"

# Define save directory for loss curves
save_dir_results = '/.../results/diffusion_model_pressure_buildup_results'
os.makedirs(save_dir_results, exist_ok=True)

# Full training with best hyperparameters
training_loss, validation_loss, test_loss = trainer_diffusion(
        model_unet=model_unet,
        vqvae_model=vqvae_model,
        ddpm=ddpm,
        criterion=criterion,
        optimizer=optimizer,
        training_dataloader=training_dataloader,
        validation_dataloader=validation_dataloader,
        testing_dataloader=testing_dataloader,
        n_epochs=n_epochs,
        save_dir=final_save_dir,
        save_dir_results=save_dir_results,
        summary_file_path=final_summary_path,
        trial_params=best_trial_params,
        save_loss_curves=True,
        lambda_pixel=1.0,
        early_stopping_patience=10  
)

# Save the final trained model along with initialization parameter (ddpm)
torch.save({
    "ddpm": ddpm,
    "state_dict": model_unet.state_dict()
}, os.path.join(final_save_dir, "diffusion_model.pth"))

# Save the final optimizer
torch.save(optimizer.state_dict(), os.path.join(final_save_dir, "diffusion_optimizer.pth"))
print("Final trained model and optimizer saved.")

### How to Activate TensorBoard ###
#Once traning is started copy paste this line into your terminal: tensorboard --logdir=/.../results/diffusion_model_pressure_buildup_results/tensorboard_logs --port=6002
#Go to VSC Command Palette and then “Simple Browser: Show” and copy paste the URL link: http://127.0.0.1:6002

#### 03b. Save Gaussian Forward Diffusion Samples

In [ ]:
# Define parameters 
sample_indices = [0, 1, 2]
t_steps=torch.linspace(start=0, end=1000-1, steps=11).long()
save_dir='/.../results/diffusion_model_pressure_buildup_results/gaussian_forward_diffusion_training_dataset'

In [ ]:
# Load pre-trained VQVAE checkpoint
vqvae_checkpoint = '/.../trained_models/pressure_buildup_models/vqvae/best_model/vqvae_model.pth'
vqvae_checkpoint = torch.load(vqvae_checkpoint)
# Extract parameters and weights
z_channels = vqvae_checkpoint["z_channels"]
codebook_size = vqvae_checkpoint["codebook_size"]
vqvae_state_dict = vqvae_checkpoint["state_dict"]
# Initialize the model with the saved parameters
vqvae_model = VQVAE(z_channels=z_channels, codebook_size=codebook_size)
# Load the model weights
vqvae_model.load_state_dict(vqvae_state_dict)
# Print successful loading
print("Model loaded successfully with z_channels={} and codebook_size={}".format(z_channels, codebook_size))

In [ ]:
# Call Function 
visualize_gaussian_forward_diffusion_dP_train(
    ddpm=ddpm,
    dataset=training_dataset,
    sample_indices=sample_indices,
    t_steps=t_steps,
    save_dir=save_dir,
    vqvae_model=vqvae_model
)

#### 03c. Save Reverse Generation Samples  

In [55]:
# Load pre-trained Unet model checkpoint
model_unet_checkpoint = "/.../trained_models/pressure_buildup_models/diffusion_model/best_model/diffusion_model.pth"
model_unet_checkpoint = torch.load(model_unet_checkpoint)

# Extract parameters and weights
ddpm = model_unet_checkpoint["ddpm"]
model_unet_state_dict = model_unet_checkpoint["state_dict"]

# Load pre-trained VQVAE checkpoint
vqvae_checkpoint = '/.../trained_models/pressure_buildup_models/vqvae/best_model/vqvae_model.pth'
vqvae_checkpoint = torch.load(vqvae_checkpoint)
# Extract parameters and weights
z_channels = vqvae_checkpoint["z_channels"]

# Initialize diffusion model
model_unet = Conditional_Unet(t_emb_dim=256, z_channels=z_channels, image_cond=True, scalar_cond=False)

# Load the model weights
model_unet.load_state_dict(model_unet_state_dict)

# Print successful loading
print("Model loaded successfully with ddpm={}".format(ddpm))

In [56]:
# Define paths and parameters
save_dir = "/.../results/diffusion_model_pressure_buildup_results/reverse_generation_training_dataset"  # Directory to save plots

# Specify sample indices 
sample_indices = list(range(500)) 

# Define reverse diffusion timesteps
steps = [1000, 900, 800, 700, 600, 500, 400, 300, 200, 100, 1]

In [ ]:
# Call the function
visualize_reverse_generation_dP_train(
    model_unet=model_unet,
    vqvae_model=vqvae_model,
    ddpm=ddpm,
    dataset=training_dataset,
    min_train_dP=min_train_dP,
    max_train_dP=max_train_dP,
    min_train_por=min_train_por,
    max_train_por=max_train_por,
    save_dir=save_dir,
    sample_indices=sample_indices,
    steps=steps
)

## References 

**[1] Wen et al., 2002** ([Paper](https://www.sciencedirect.com/science/article/pii/S0309170822000562)): CO₂ Open-Source Dataset ([Google Drive Link](https://drive.google.com/drive/folders/1fZQfMn_vsjKUXAfRV0q_gswtl8JEkVGo)).

[2] How to train a Latent Conditional Diffusion Model ([GitHub Link](https://github.com/explainingai-code/StableDiffusion-PyTorch/tree/main)).